# Module 1 — Pharmacovigilance Signal Detection (FAERS)

**Scenario**: A pharma client wants an early-warning watchlist of drug–adverse event combinations that are reported disproportionately often, using the FDA Adverse Event Reporting System (FAERS) via openFDA.

**This notebook**:
1. Pulls real FAERS data for a drug of interest
2. Computes disproportionality statistics (PRR, ROR, chi-square) for each reported reaction vs. background rates
3. Flags reporting-volume anomalies over time
4. Produces a risk-scored signal watchlist
5. Writes a plain-English stakeholder summary to `reports/faers_summary.md`

**Important caveat (note for the report)**: openFDA explicitly states results are *unvalidated* and should not be used for medical decisions. A high PRR can also reflect **confounding by indication** — e.g. a drug prescribed for rheumatoid arthritis will show a strong 'signal' for the reaction term 'Rheumatoid arthritis' simply because patients who take it already have that condition, not because the drug causes it. Any real watchlist needs clinical review before action.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data_connectors.openfda_faers import (
    fetch_events_for_drug,
    fetch_reaction_counts,
    fetch_total_report_count,
)
from src.risk_framework.signal_detection import compute_signal
from src.risk_framework.anomaly_detection import rolling_volume_anomalies
from src.risk_framework.risk_scoring import RiskScoringEngine

DRUG = "HUMIRA"
SEARCH = f'patient.drug.medicinalproduct:"{DRUG}"'

## 1. Pull report-level data and reaction counts

In [2]:
events = fetch_events_for_drug(DRUG, max_records=3000)
print(f"Fetched {len(events)} (report, reaction) rows for {DRUG}")
events.head()

Fetched 13423 (report, reaction) rows for HUMIRA


,safetyreportid,receivedate,serious,primarysourcecountry,patientonsetage,patientsex,queried_drug,drugs_in_report,reaction_pt,reaction_outcome
0,10003532,20140312,1,GB,59,1,HUMIRA,"AMLODIPINE, HUMIRA",Chest pain,1
1,10003532,20140312,1,GB,59,1,HUMIRA,"AMLODIPINE, HUMIRA",Oedema peripheral,1
2,10003532,20140312,1,GB,59,1,HUMIRA,"AMLODIPINE, HUMIRA",Gastrooesophageal reflux disease,1
3,10003540,20140312,1,US,None,1,HUMIRA,HUMIRA,Oesophageal perforation,6
4,10003540,20140312,1,US,None,1,HUMIRA,HUMIRA,Gastric haemorrhage,6


In [3]:
# Top reaction terms reported with this drug, plus totals needed for the 2x2 tables
drug_reactions = fetch_reaction_counts(search=SEARCH, limit=20)
total_drug = fetch_total_report_count(search=SEARCH)
total_all = fetch_total_report_count()

print(f"Total FAERS reports mentioning {DRUG}: {total_drug:,}")
print(f"Total FAERS reports overall: {total_all:,}")
drug_reactions.head(10)

Total FAERS reports mentioning HUMIRA: 685,851
Total FAERS reports overall: 20,328,575


,reaction_pt,count
0,DRUG INEFFECTIVE,86367
1,INJECTION SITE PAIN,50447
2,PAIN,48259
3,ARTHRALGIA,47223
4,FATIGUE,38208
5,RHEUMATOID ARTHRITIS,34046
6,HEADACHE,27871
7,NAUSEA,26425
8,PSORIASIS,24564
9,RASH,24485


## 2. Disproportionality analysis (PRR / ROR / chi-square)

For each of the drug's top reaction terms, build the 2x2 contingency table:

|                | Reaction = X | Reaction != X |
|----------------|--------------|----------------|
| Drug = DRUG    | a            | b              |
| Drug != DRUG   | c            | d              |

`c` requires the *total* number of FAERS reports (any drug) with reaction X — one extra API call per reaction term, cached locally.

In [4]:
# Some reaction terms contain characters (e.g. "^", "'") that the openFDA query
# parser rejects — skip those terms rather than failing the whole pipeline, and
# report how many were skipped (a real example of messy regulatory data).
signal_results = []
skipped_terms = []
for _, row in drug_reactions.iterrows():
    pt = row["reaction_pt"]
    a = int(row["count"])
    b = total_drug - a
    try:
        total_event = fetch_total_report_count(search=f'patient.reaction.reactionmeddrapt.exact:"{pt}"')
    except Exception:
        skipped_terms.append(pt)
        continue
    c = total_event - a
    d = total_all - a - b - c
    signal_results.append(compute_signal(a, b, c, d, drug=DRUG, event=pt))

if skipped_terms:
    print(f"Skipped {len(skipped_terms)} reaction term(s) with unparsable characters: {skipped_terms}")

records = [{**s.__dict__, "meets_signal_threshold": s.meets_signal_threshold} for s in signal_results]
signals_df = pd.DataFrame(records).sort_values("prr", ascending=False)
signals_df[["event", "a", "prr", "prr_ci_low", "prr_ci_high", "ror", "chi_square", "meets_signal_threshold"]]

Skipped 1 reaction term(s) with unparsable characters: ['CROHN^S DISEASE']


,event,a,prr,prr_ci_low,prr_ci_high,ror,chi_square,meets_signal_threshold
5,RHEUMATOID ARTHRITIS,34046,12.881290,12.720495,13.044118,13.501891,258745.221525,True
18,INJECTION SITE HAEMORRHAGE,16850,8.066439,7.931076,8.204113,8.244420,81688.432272,True
17,DEVICE ISSUE,17992,7.853588,7.726474,7.982792,8.038222,84793.912119,True
13,JOINT SWELLING,22652,6.889154,6.791317,6.988401,7.090302,92451.549695,True
8,PSORIASIS,24564,6.869086,6.775519,6.963946,7.087098,99971.061178,True
1,INJECTION SITE PAIN,50447,5.992603,5.937194,6.048528,6.388983,176045.298428,True
3,ARTHRALGIA,47223,3.824515,3.789127,3.860234,4.033373,88647.277798,True
14,NASOPHARYNGITIS,20730,3.744738,3.691665,3.798574,3.830284,37207.416216,True
12,INCORRECT DOSE ADMINISTERED,23188,3.439147,3.393343,3.485569,3.524498,36195.676920,True
11,PAIN IN EXTREMITY,24112,2.620923,2.587160,2.655126,2.679985,22461.181744,True


## 3. Reporting-volume anomalies over time

Spikes in monthly report volume can indicate a media event, label change, litigation, or a genuine emerging safety issue.

In [5]:
volume = rolling_volume_anomalies(events, date_col="receivedate", freq="M", threshold=2.5)
anomalous_months = volume[volume["is_anomaly"]]
print(f"{len(anomalous_months)} anomalous month(s) out of {len(volume)}")
anomalous_months

0 anomalous month(s) out of 7


,count,is_anomaly,zscore
receivedate,,,


## 4. Risk-scored watchlist

Combine three normalised (0-1) factors per reaction term into a single risk score:
- **prr_strength**: PRR normalised against a cap (PRR of 10+ -> 1.0)
- **volume**: report count normalised against the largest reaction's count
- **statistical_confidence**: 1 if chi-square >= 4 (statistically notable), else scaled down

In [6]:
scored = signals_df.copy()
scored["prr_strength"] = (scored["prr"] / 10).clip(upper=1.0)
scored["volume"] = (scored["a"] / scored["a"].max()).clip(upper=1.0)
scored["statistical_confidence"] = (scored["chi_square"] / 4).clip(upper=1.0)

engine = RiskScoringEngine(weights={"prr_strength": 0.5, "volume": 0.2, "statistical_confidence": 0.3})
scored = engine.score_dataframe(scored)
watchlist = scored.sort_values("risk_score", ascending=False)
watchlist[["event", "prr", "a", "risk_score", "risk_band"]]

,event,prr,a,risk_score,risk_band
5,RHEUMATOID ARTHRITIS,12.881290,34046,0.8788,Critical
18,INJECTION SITE HAEMORRHAGE,8.066439,16850,0.7423,High
17,DEVICE ISSUE,7.853588,17992,0.7343,High
1,INJECTION SITE PAIN,5.992603,50447,0.7165,High
8,PSORIASIS,6.869086,24564,0.7003,High
13,JOINT SWELLING,6.889154,22652,0.6969,High
0,DRUG INEFFECTIVE,2.072763,86367,0.6036,High
3,ARTHRALGIA,3.824515,47223,0.6006,High
2,PAIN,2.508411,48259,0.5372,High
14,NASOPHARYNGITIS,3.744738,20730,0.5352,High


## 6. Report-level anomaly detection model (Isolation Forest)

The disproportionality analysis above looks at *reaction terms* in aggregate. A
complementary approach used in real pharmacovigilance teams is **case-level
triage**: flag individual reports that look statistically unusual given their
overall profile (number of reactions, number of co-reported drugs, patient age,
seriousness), so that signal-management staff can prioritise which case
narratives to read first.

We aggregate the (report, reaction) rows to one row per report, train an
`IsolationForest` (unsupervised — no labels exist for "this case deserves a
closer look"), and persist the fitted model plus its feature list and training
metadata to `models/` for reuse.

In [7]:
import joblib
from datetime import datetime, timezone
from pathlib import Path
from sklearn.ensemble import IsolationForest

# Aggregate (report, reaction) rows to one row per report
reports = events.groupby("safetyreportid").agg(
    receivedate=("receivedate", "first"),
    serious=("serious", "first"),
    patientonsetage=("patientonsetage", "first"),
    patientsex=("patientsex", "first"),
    n_reactions=("reaction_pt", "count"),
    drugs_in_report=("drugs_in_report", "first"),
).reset_index()

reports["patientonsetage"] = pd.to_numeric(reports["patientonsetage"], errors="coerce")
age_median = reports["patientonsetage"].median()
reports["patientonsetage_imputed"] = reports["patientonsetage"].fillna(age_median)
reports["n_drugs"] = reports["drugs_in_report"].str.split(",").str.len()
reports["serious"] = pd.to_numeric(reports["serious"], errors="coerce").fillna(0)

pct_missing_age = reports["patientonsetage"].isna().mean()
print(f"{len(reports)} unique reports. {pct_missing_age:.1%} missing patient age "
      f"(imputed with median={age_median:.0f}).")

FEATURES = ["patientonsetage_imputed", "n_reactions", "n_drugs", "serious"]
CONTAMINATION = 0.05

iforest = IsolationForest(contamination=CONTAMINATION, random_state=42)
reports["anomaly_flag"] = iforest.fit_predict(reports[FEATURES]) == -1
reports["anomaly_score"] = -iforest.score_samples(reports[FEATURES])

n_anomalies = int(reports["anomaly_flag"].sum())
print(f"Flagged {n_anomalies} of {len(reports)} reports ({n_anomalies / len(reports):.1%}) "
      f"as case-level anomalies (contamination={CONTAMINATION}).")
reports.sort_values("anomaly_score", ascending=False)[
    ["safetyreportid", "receivedate", "serious", "patientonsetage_imputed", "n_reactions", "n_drugs", "anomaly_score"]
].head(10)

3000 unique reports. 33.3% missing patient age (imputed with median=52).


Flagged 150 of 3000 reports (5.0%) as case-level anomalies (contamination=0.05).


,safetyreportid,receivedate,serious,patientonsetage_imputed,n_reactions,n_drugs,anomaly_score
1077,10073807,20140411,1,52.0,36,26,0.755342
1360,10098857,20140423,1,36.0,42,21,0.749085
1966,10162188,20140509,1,54.0,34,24,0.743679
2287,10177773,20140516,1,43.0,19,31,0.735217
1709,10149947,20140502,1,52.0,26,26,0.734677
323,10027859,20140321,1,70.0,25,26,0.734108
2570,10194046,20140524,2,75.0,1,22,0.733828
2725,10208625,20140528,2,79.0,1,20,0.730855
1090,10074111,20140411,1,50.0,21,32,0.729765
1152,10079193,20140414,1,77.0,13,34,0.714420


In [8]:
# Persist the fitted model + metadata for reuse (e.g. scoring new report batches)
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "faers_case_anomaly_iforest.joblib"
joblib.dump({"model": iforest, "features": FEATURES, "age_median": age_median}, model_path)

metadata = {
    "model": "IsolationForest",
    "purpose": "Case-level anomaly flagging for FAERS reports (signal-management triage)",
    "drug": DRUG,
    "features": FEATURES,
    "contamination": CONTAMINATION,
    "age_median_imputed": float(age_median),
    "training_rows": len(reports),
    "n_flagged": n_anomalies,
    "anomaly_rate": n_anomalies / len(reports),
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "caveats": [
        "Unsupervised: there is no ground-truth label for 'this report deserves review', "
        "so 'anomaly' means 'statistically unusual given this drug's typical report "
        "profile', not 'confirmed safety issue'.",
        "contamination=0.05 is an analyst-chosen assumption (expect ~5% of reports to be "
        "flagged), not derived from data.",
        f"{pct_missing_age:.1%} of reports were missing patient age and had it imputed "
        "with the cohort median — flagged reports with imputed age should be interpreted "
        "with that in mind.",
    ],
}
with open(models_dir / "faers_case_anomaly_iforest.meta.json", "w", encoding="utf-8") as f:
    import json
    json.dump(metadata, f, indent=2)

print(f"Saved model to {model_path.resolve()}")
print(f"Saved metadata to {(models_dir / 'faers_case_anomaly_iforest.meta.json').resolve()}")

Saved model to C:\Users\allif\Downloads\Risk Modelling\models\faers_case_anomaly_iforest.joblib
Saved metadata to C:\Users\allif\Downloads\Risk Modelling\models\faers_case_anomaly_iforest.meta.json


## 7. Save processed outputs

Persist the scored signal watchlist and case-level anomaly table to
`data/processed/faers/` for downstream use (e.g. the Streamlit dashboard),
alongside a metadata sidecar.

In [9]:
processed_dir = Path("../data/processed/faers")
processed_dir.mkdir(parents=True, exist_ok=True)

watchlist.to_csv(processed_dir / "signal_watchlist.csv", index=False)
reports.to_csv(processed_dir / "case_anomalies.csv", index=False)

with open(processed_dir / "processed.meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "drug": DRUG,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "files": {
            "signal_watchlist.csv": f"{len(watchlist)} reaction-term signals, risk-scored",
            "case_anomalies.csv": f"{len(reports)} reports with Isolation Forest anomaly scores",
        },
    }, f, indent=2)

print(f"Wrote {len(watchlist)} signal rows and {len(reports)} case rows to {processed_dir.resolve()}")

Wrote 19 signal rows and 3000 case rows to C:\Users\allif\Downloads\Risk Modelling\data\processed\faers


## 8. Write stakeholder summary

In [10]:
from datetime import datetime, timezone
from pathlib import Path

top5 = watchlist.head(5)

lines = []
lines.append(f"# Pharmacovigilance Signal Watchlist — {DRUG}\n")
lines.append(f"_Generated {datetime.now(timezone.utc).date()} from openFDA FAERS data "
              f"({total_drug:,} reports for {DRUG} out of {total_all:,} total reports)._\n")
lines.append("## What this means\n")
lines.append(f"Out of {DRUG}'s most commonly reported adverse events, "
              f"{len(scored[scored['meets_signal_threshold']])} are reported "
              f"disproportionately more often than for drugs in general. "
              f"The table below ranks these by overall risk (statistical strength, "
              f"reporting volume, and reporting confidence combined).\n")
lines.append("## Top signals\n")
lines.append("| Reaction | Reports | PRR (95% CI) | Risk band |")
lines.append("|---|---|---|---|")
for _, r in top5.iterrows():
    lines.append(f"| {r['event'].title()} | {r['a']:,} | "
                  f"{r['prr']:.1f} ({r['prr_ci_low']:.1f}-{r['prr_ci_high']:.1f}) | {r['risk_band']} |")
lines.append("")
lines.append("## Important caveats\n")
lines.append("- openFDA data is **unvalidated** spontaneous-report data — it shows reporting "
              "patterns, not confirmed causation.\n")
lines.append("- A high signal for a condition the drug *treats* (e.g. an arthritis drug showing "
              "a signal for 'Rheumatoid arthritis') usually reflects **confounding by "
              "indication**, not a side effect — these need clinical triage, not automatic "
              "escalation.\n")
lines.append(f"- {len(anomalous_months)} month(s) showed anomalous reporting volume — worth "
              "checking against label changes, litigation, or media coverage in those periods.\n")
lines.append("## Case-level anomaly triage\n")
lines.append(f"An Isolation Forest model trained on {len(reports):,} individual {DRUG} reports "
              f"(features: number of reactions, number of co-reported drugs, patient age, "
              f"seriousness) flagged {n_anomalies} reports ({n_anomalies / len(reports):.1%}) "
              f"as statistically unusual case profiles. These are candidates for case-narrative "
              f"review by signal-management staff, not confirmed issues. Model saved to "
              f"`models/faers_case_anomaly_iforest.joblib`.\n")
lines.append("## Recommended next steps\n")
lines.append("1. Clinical reviewer to triage the top-ranked signals, separating known "
              "indications/contraindications from genuinely new signals.\n")
lines.append("2. Signal-management staff to review the highest-scoring case-level anomalies "
              "identified by the Isolation Forest model.\n")
lines.append("3. Re-run this analysis monthly to track trend direction (worsening vs. stable).\n")
lines.append("4. For any new genuine signal, cross-check against the product's current label "
              "and any open regulatory safety communications.\n")

report_path = Path("../reports/faers_summary.md")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {report_path.resolve()}")

Wrote C:\Users\allif\Downloads\Risk Modelling\reports\faers_summary.md
